# Рекомендательные системы на основе коллаборативной фильтрации




In [25]:
%pip install --upgrade ipywidgets jupyter

  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
    --------------------------------------- 0.3/17.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/17.2 MB 2.4 MB/s eta 0:00:07
   -- ------------------------------------- 1.0/17.2 MB 2.1 MB/s eta 0:00:08
   --- ------------------------------------ 1.6/17.2 MB 2.2 MB/s eta 0:00:08
   ---- ----------------------------------- 1.8/17.2 MB 2.2 MB/s eta 0:00:07
   ------ --------------------------------- 2.6/17.2 MB 2.2 MB/s eta 0:00:07
   ------ --------------------------------- 2.9/17.2 MB 2.2 MB/s eta 0:00:07
   -------- -------------------------------

In [23]:
%pip install --upgrade tqdm ipywidgets

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 2.1 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.2 MB 2.5 MB/s eta 0:00:01
   -------------------------------- ------- 1.8/2.2 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 2.7 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


In [1]:
import warnings
warnings.simplefilter('ignore')

import pandas as pd
import numpy as np
from tqdm import tqdm_notebook
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.base import BaseEstimator
import kagglehub

%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib




Рассмотрим построение рекомендательной системы на датасете от `GroupLens` $-$ [`MovieLens`](https://grouplens.org/datasets/movielens/):
Это набор данных из $9 000$ фильмов и $700$ пользователей, с общим количеством оценок в $100 000$.


Скачать напрямую датасет можно по этой [ссылке](http://files.grouplens.org/datasets/movielens/ml-latest-small.zip)

**Загрузка данных**

## Данные
 - `userID`
 - `songID`
 - `rating`

In [2]:
ratings = pd.read_csv("data/songsDataset.csv")
ratings.head()

,'userID','songID','rating'
0,0,7171,5
1,0,8637,4
2,0,21966,4
3,0,35821,5
4,0,82446,5


In [3]:
ratings.value_counts()

'userID'  'songID'  'rating'
0         7171      5           1
          8637      4           1
          21966     4           1
          35821     5           1
          82446     5           1
                               ..
199999    75840     2           1
          85921     1           1
          116763    1           1
          128758    1           1
          129321    1           1
Name: count, Length: 2000000, dtype: int64

### Метрика. Формирование тестовой выборки.

Используется `RMSE`: классическая метрика для задач рекомендации после прошедшего [Netflix Prize](https://ru.wikipedia.org/wiki/Netflix_Prize).

Выделим часть выборки для тестирования модели по принципу: для каждого пользователя возьмем последние 20% оценок.

In [4]:
rmse = lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))

def train_test_split(X, ratio=0.2, user_col='\'userID\'', item_col='\'songID\'',
                     rating_col='\'rating\''):
    # список всех юзеров
    userIds = X[user_col].unique()
    X_train_data = []
    X_test_data = []
    y_train = []
    y_test = []
    for userId in tqdm_notebook(userIds):
        curUser = X[X[user_col] == userId]
        # определяем позицию, по которой делим выборку и размещаем данные по массивам
        idx = int(curUser.shape[0] * (1 - ratio))
        X_train_data.append(curUser[user_col].iloc[:idx].values)
        X_test_data.append(curUser[user_col].iloc[idx:].values)
        y_train.append(curUser[rating_col].values[:idx])
        y_test.append(curUser[rating_col].values[idx:])
    # cтекуем данные по каждому пользователю в общие массивы
    X_train = pd.DataFrame(np.vstack(X_train_data), columns=[user_col, item_col])
    X_test = pd.DataFrame(np.vstack(X_test_data), columns=[user_col, item_col])
    y_train = np.hstack(y_train)
    y_test = np.hstack(y_test)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = train_test_split(ratings)

  0%|          | 0/200000 [00:00<?, ?it/s]

ValueError: Shape of passed values is (200000, 8), indices imply (200000, 2)

In [16]:
X_train.shape, len(y_train), X_test.shape, len(y_test)

NameError: name 'X_train' is not defined

In [ ]:
X_train.userId.nunique()

610

# Корреляционные модели
#



Требуется реализовать User-based и Item-based модели

# Surprise
Библиотека [Surprise](http://surpriselib.com/) .


In [ ]:
%pip install surprise

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.0/772.0 KB 11.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.3-cp39-cp39-linux_x86_64.whl size=3193639 sha256=6ebff6da9a5a5d6af4da843c370b3d68743bf35b467ffc3c889717665146b2a0
  Stored in directory: /root/.cache/pip/wheels/c6/3a/46/9b17b3512bdf283c6cb84f59929cdd5199d4e754d596d22784
Successfully built scikit-surprise


In [ ]:
from surprise.prediction_algorithms.knns import KNNBasic
from surprise import Dataset
from surprise.model_selection import cross_validate

# Load the movielens-100k dataset (download it if needed).
data = Dataset.load_builtin('ml-100k')

# Use the user-based algorithm.
algo = KNNBasic()

# Run 5-fold cross-validation and print results.
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)


Dataset ml-100k could not be found. Do you want to download it? [Y/n] Y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBasic on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9791  0.9906  0.9761  0.9715  0.9776  0.9790  0.0064  
MAE (testset)     0.7725  0.7834  0.7720  0.7654  0.7732  0.7733  0.0058  
Fit time          0.30    0.39    0.34    0.34    0.51    0.38    0.07    
Test time         6.34    4.43    3.36   

{'test_rmse': array([0.97907514, 0.99060729, 0.97610275, 0.97145288, 0.97763558]),
 'test_mae': array([0.77249332, 0.78344244, 0.77202184, 0.76535443, 0.77322068]),
 'fit_time': (0.3007166385650635,
  0.38873767852783203,
  0.3420846462249756,
  0.3421168327331543,
  0.5058658123016357),
 'test_time': (6.341052532196045,
  4.432601690292358,
  3.356508493423462,
  3.5385327339172363,
  4.17162299156189)}